# Analisi preliminare e selezione del sottoinsieme d'interesse

Laura Pala, https://github.com/LP-fw/Anomaly-Detection-progetto-Pala

Questo notebook documenta la derivazione del file usato nel progetto `greenhouse_definitivo.csv` a partire dal dataset originale `Greenhouse-1.xlsx`, disponibile su Zenodo (DOI 10.5281/zenodo.5793685).

L'originale contiene i dati di 27 sensori (AF16–AF42) in fogli separati.

Se il file è già presente in locale, il download viene saltato!

In [15]:
#librerie necessarie a lettura, scaricamento e manipolazione
import pandas as pd
from pathlib import Path
import requests

In [16]:
URL = 'https://zenodo.org/records/5793685/files/Greenhouse-1.xlsx?download=1'
FILEPATH = Path('Greenhouse-1.xlsx')

if not FILEPATH.exists():
    print('Download da Zenodo...')
    r = requests.get(URL, timeout=300)
    r.raise_for_status()
    FILEPATH.write_bytes(r.content)
    print(f'Scaricato: {FILEPATH.stat().st_size / 1e6:.1f} MB')
else:
    print(f'File già presente in locale: {FILEPATH}')

File già presente in locale: Greenhouse-1.xlsx


## Selezione dei sensori

Dei 27 sensori del deployment (AF16-AF42), il notebook principale utilizza il sottoinsieme AF26, AF27, AF34, AF36, AF39.

AF27 per l'anomalia di misura del 14 luglio 2020,

AF39 come sensore a variabilità elevata,

AF26, AF34, AF36 come sensori a comportamento apparentemente regolare.

In [17]:
sensori = ['AF27', 'AF39', 'AF34', 'AF26', 'AF36']

dfs = []
for sheet in sensori:
    df = pd.read_excel(FILEPATH, sheet_name=sheet)
    df['sensor_id'] = sheet
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Righe totali dopo caricamento: {len(df_all)}")

# Timestamp robusto: split su Date + errors='coerce'
df_all['timestamp'] = pd.to_datetime(
    df_all['Date'].astype(str).str.split(' ').str[0] + ' ' + df_all['Time'].astype(str),
    errors='coerce'
)

# Scarto righe senza timestamp valido
n_prima = len(df_all)
df_all = df_all.dropna(subset=['timestamp']).reset_index(drop=True)
print(f"Righe scartate: {n_prima - len(df_all)}")
print(f"Righe rimaste: {len(df_all)}")

Righe totali dopo caricamento: 149302
Righe scartate: 1
Righe rimaste: 149301


In [18]:
df_all.head(10)

,Date,Time,Onboard Temperature,Onboard Humidity,RSSI,SNR,sensor_id,timestamp
0,2020-04-10,13:57:09,27.598,28.099,-82,10.2,AF27,2020-04-10 13:57:09
1,2020-04-10,14:07:27,27.585,27.805,-87,13.2,AF27,2020-04-10 14:07:27
2,2020-04-10,14:12:35,27.614,27.372,-88,13.5,AF27,2020-04-10 14:12:35
3,2020-04-10,14:22:52,27.556,26.766,-85,12.5,AF27,2020-04-10 14:22:52
4,2020-04-10,14:28:01,27.572,26.740,-84,11.5,AF27,2020-04-10 14:28:01
5,2020-04-10,14:33:10,27.542,26.653,-83,9.8,AF27,2020-04-10 14:33:10
6,2020-04-10,14:38:18,27.515,26.889,-87,9.8,AF27,2020-04-10 14:38:18
7,2020-04-10,14:43:27,27.459,26.999,-87,13.2,AF27,2020-04-10 14:43:27
8,2020-04-10,13:46:52,27.641,27.655,-83,12.8,AF27,2020-04-10 13:46:52
9,2020-04-10,13:52:01,27.628,27.779,-84,12.5,AF27,2020-04-10 13:52:01


In [19]:
df_all.describe()

,Date,Onboard Temperature,Onboard Humidity,RSSI,SNR,timestamp
count,149301,149301.000000,149301.000000,149301.000000,149301.000000,149301
mean,2020-06-23 10:15:33.003797504,22.376976,63.934707,-95.278679,4.450885,2020-06-23 22:20:04.395449344
min,2020-04-10 00:00:00,-45.000000,0.105000,-126.000000,-15.500000,2020-04-10 13:46:26
25%,2020-05-12 00:00:00,18.452000,52.979000,-114.000000,-2.800000,2020-05-12 04:35:43
50%,2020-06-21 00:00:00,21.961000,65.409000,-107.000000,7.200000,2020-06-21 22:12:07
75%,2020-07-22 00:00:00,25.689000,76.831000,-75.000000,12.200000,2020-07-22 01:06:31
max,2020-11-11 00:00:00,41.382000,100.000000,-51.000000,14.800000,2020-11-11 08:23:26
std,NaN,5.081246,16.551807,19.682922,8.337714,NaN


In [20]:
CSV_OUT = Path('greenhouse_definitivo.csv')
df_all.to_csv(CSV_OUT, index=False)
print(f'Salvato: {CSV_OUT} ({CSV_OUT.stat().st_size / 1e6:.2f} MB, {len(df_all):,} righe)')

Salvato: greenhouse_definitivo.csv (10.16 MB, 149,301 righe)


In [21]:
# check del csv appena salvato
check = pd.read_csv(CSV_OUT)
print(check.shape)
print(check.dtypes)
check.head(3)

(149301, 8)
Date                    object
Time                    object
Onboard Temperature    float64
Onboard Humidity       float64
RSSI                     int64
SNR                    float64
sensor_id               object
timestamp               object
dtype: object


,Date,Time,Onboard Temperature,Onboard Humidity,RSSI,SNR,sensor_id,timestamp
0,2020-04-10,13:57:09,27.598,28.099,-82,10.2,AF27,2020-04-10 13:57:09
1,2020-04-10,14:07:27,27.585,27.805,-87,13.2,AF27,2020-04-10 14:07:27
2,2020-04-10,14:12:35,27.614,27.372,-88,13.5,AF27,2020-04-10 14:12:35
